# Interactive Testing of ReportOrchestrator
This notebook allows you to test the `ReportOrchestrator` using the real data files located in `private_file/`.

In [6]:
import os
import sys
import django

# Setup Django environment
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()
print("Django environment configured.")

Django environment configured.


In [7]:
import json
from pathlib import Path
from django.conf import settings
from core.llm_provider import LLMProvider
from core.report_orchestrator import ReportOrchestrator

# Note: The LLMProvider requires the GROQ_API_KEY environment variable.
# Make sure it's defined in your environment or in backend/.env file.
try:
    llm_provider = LLMProvider(api_key=settings.GROQ_API_KEY)
    orchestrator = ReportOrchestrator(llm_provider=llm_provider)
    print("Orchestrator initialized.")
except Exception as e:
    print(f"Failed to initialize orchestrator: {e}")

Orchestrator initialized.


In [8]:
# Choose a data file from private_file directory
data_file_path = Path("../private_file/billing_log_2026-07-27.json")
with open(data_file_path, "r") as f:
    raw_rows = json.load(f)

print(f"Loaded {len(raw_rows)} rows from {data_file_path.name}")

Loaded 19 rows from billing_log_2026-07-27.json


In [9]:
clinic_id = "CLN-INTERACTIVE-01"
batch_name = data_file_path.name

print("Generating daily report...")
response = orchestrator.generate_daily_report(
    clinic_id=clinic_id,
    raw_rows=raw_rows,
    batch=batch_name
)

print("\n--- Ingestion Results ---")
print(f"Total Rows Processed: {response['total_rows_processed']}")
print(f"Valid Records Count: {response['valid_records_count']}")
print(f"Errors Count: {len(response['errors'])}")

Generating daily report...

--- Ingestion Results ---
Total Rows Processed: 19
Valid Records Count: 18
Errors Count: 1


In [10]:
print("\n--- Validation Errors (Top 5) ---")
for i, err in enumerate(response["errors"][:5]):
    print(f"{i+1}. Row: {err.row_ref} | Field: {err.field} | Reason: {err.reason}")
if len(response["errors"]) > 5:
    print(f"... and {len(response['errors']) - 5} more errors.")


--- Validation Errors (Top 5) ---
1. Row: V-20260727-019 | Field: payment_mode | Reason: missing or wrong type


In [11]:
recon = response["recon_report"]
print("\n--- Reconciliation Report ---")
if recon:
    print(f"Total Billed (Paise): {recon.total_billed_paise}")
    print(f"Total Collected (Paise): {recon.total_collected_paise}")
    print(f"Total Outstanding (Paise): {recon.total_outstanding_paise}")
    print(f"Total Refunds (Paise): {recon.total_refunds_paise}")
    print(f"Visit Count: {recon.visit_count}")
    print(f"Refund Count: {recon.refund_count}")
else:
    print("No valid records to process.")


--- Reconciliation Report ---
Total Billed (Paise): 326000
Total Collected (Paise): 317200
Total Outstanding (Paise): 8800
Total Refunds (Paise): 0
Visit Count: 18
Refund Count: 0


In [12]:
analytics = response["analytics_report"]
print("\n--- Analytics Report (Revenue by Hour) ---")
if analytics:
    for hourly in analytics.revenue_by_hour.all():
        print(f"Hour {hourly.hour:02d}: {hourly.revenue_paise} paise")
else:
    print("No analytics data.")


--- Analytics Report (Revenue by Hour) ---
Hour 09: 9000 paise
Hour 10: 56500 paise
Hour 11: 33500 paise
Hour 12: 9500 paise
Hour 13: 75500 paise
Hour 14: 3500 paise
Hour 15: 41500 paise
Hour 16: 60200 paise
Hour 17: 22000 paise
Hour 18: 6000 paise


In [13]:
narrative = response["narrative_result"]
print("\n--- Generated Narrative ---")
if narrative:
    print(f"Status: {narrative.status}")
    print(f"Warnings: {narrative.warnings}")
    print("\nText:")
    print(narrative.text)
else:
    print("No narrative generated.")


--- Generated Narrative ---
Status: SUCCESS
Warnings: []

Text:
Billing Summary for 18 Visits
---------------------------
Total Billed: $₹3,260
Total Collected: $₹3,172
Total Outstanding: $₹88
Total Refunds: $₹0
Refund Count: 0
Peak Hour Revenue: $₹755 at 1pm–2pm
Top Medicine by Quantity: OMEPRAZOLE (18 units)
Top Medicine by Revenue: ATORVASTATIN (₹1,200)
---------------------------
Note: The figures above are subject to change based on ongoing transactions.
Note: cost data wasn't available today, so this is revenue, not profit.
